In [ ]:
import numpy as np

In [ ]:
class MLPClassifier:
    def __init__(self, hidden_layers=[8, 8], learning_rate=0.01, epochs=5000, activation="relu", random_state=42):
        self.hidden_layers = hidden_layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.activation = activation
        self.random_state = random_state
        self.params = {}
        self.loss_history = []
        self.classes_ = None
        self.n_features_ = None
        self.n_outputs_ = None

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y)

        self.classes_ = np.unique(y)
        self.n_features_ = X.shape[1]
        self.n_outputs_ = len(self.classes_)

        y_encoded = self._one_hot(y)
        self._initialize_parameters()

        for _ in range(self.epochs):
            activations, zs = self._forward(X)
            loss = self._cross_entropy(y_encoded, activations[-1])
            self.loss_history.append(loss)
            grads = self._backward(X, y_encoded, activations, zs)
            self._update_parameters(grads)

    def predict(self, X):
        X = np.array(X, dtype=float)
        activations, _ = self._forward(X)
        y_pred_idx = np.argmax(activations[-1], axis=1)
        return self.classes_[y_pred_idx]

    def predict_proba(self, X):
        X = np.array(X, dtype=float)
        activations, _ = self._forward(X)
        return activations[-1]

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == np.array(y))

    def _initialize_parameters(self):
        np.random.seed(self.random_state)
        layer_sizes = [self.n_features_] + self.hidden_layers + [self.n_outputs_]

        for i in range(1, len(layer_sizes)):
            input_size = layer_sizes[i - 1]
            output_size = layer_sizes[i]

            if i < len(layer_sizes) - 1 and self.activation == "relu":
                scale = np.sqrt(2 / input_size)
            else:
                scale = np.sqrt(1 / input_size)

            self.params[f"W{i}"] = np.random.randn(input_size, output_size) * scale
            self.params[f"b{i}"] = np.zeros((1, output_size))

    def _forward(self, X):
        activations = [X]
        zs = []
        A = X
        L = len(self.hidden_layers) + 1

        for i in range(1, L):
            Z = A @ self.params[f"W{i}"] + self.params[f"b{i}"]
            A = self._activate(Z)
            zs.append(Z)
            activations.append(A)

        Z = A @ self.params[f"W{L}"] + self.params[f"b{L}"]
        A = self._softmax(Z)
        zs.append(Z)
        activations.append(A)

        return activations, zs

    def _backward(self, X, y_true, activations, zs):
        grads = {}
        m = X.shape[0]
        L = len(self.hidden_layers) + 1

        dZ = activations[-1] - y_true
        grads[f"W{L}"] = activations[-2].T @ dZ / m
        grads[f"b{L}"] = np.sum(dZ, axis=0, keepdims=True) / m

        for i in range(L - 1, 0, -1):
            dA = dZ @ self.params[f"W{i + 1}"].T
            dZ = dA * self._activate_derivative(zs[i - 1])
            grads[f"W{i}"] = activations[i - 1].T @ dZ / m
            grads[f"b{i}"] = np.sum(dZ, axis=0, keepdims=True) / m

        return grads

    def _update_parameters(self, grads):
        for key in grads:
            self.params[key] -= self.learning_rate * grads[key]

    def _activate(self, Z):
        if self.activation == "sigmoid":
            return 1 / (1 + np.exp(-Z))
        if self.activation == "tanh":
            return np.tanh(Z)
        return np.maximum(0, Z)

    def _activate_derivative(self, Z):
        if self.activation == "sigmoid":
            A = 1 / (1 + np.exp(-Z))
            return A * (1 - A)
        if self.activation == "tanh":
            return 1 - np.tanh(Z) ** 2
        return (Z > 0).astype(float)

    def _softmax(self, Z):
        Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
        exp_values = np.exp(Z_shifted)
        return exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def _cross_entropy(self, y_true, y_pred):
        eps = 1e-12
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    def _one_hot(self, y):
        y_idx = np.array([np.where(self.classes_ == label)[0][0] for label in y])
        one_hot = np.zeros((len(y), len(self.classes_)))
        one_hot[np.arange(len(y)), y_idx] = 1
        return one_hot

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y = np.array([0, 1, 1, 0, 0, 1, 1, 0])

In [ ]:
mlp = MLPClassifier(
    hidden_layers=[8, 8],
    learning_rate=0.1,
    epochs=5000,
    activation="relu",
    random_state=42
)

mlp.fit(X, y)

predictions = mlp.predict(X)
accuracy = mlp.score(X, y)

print("Predictions:", predictions)
print("Accuracy:", accuracy)

In [ ]:
print(mlp.predict_proba(X))

In [ ]:
print(mlp.loss_history[:10])
print(mlp.loss_history[-10:])

In [ ]:
X_new = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

print(mlp.predict(X_new))
print(mlp.predict_proba(X_new))